### 02_silveringestion.ipynb
* Silver Layer — eCourts India Pipeline
* Flow: Bronze/cases + Bronze/courts → silver_cleaned → silver_enriched
* Silver Layer : \
        - `silver_cleaned` : cast types, remove nulls, dedupe on case_id, keep is_deleted flag \
        - `silver_enriched`: filter is_deleted=false, INNER JOIN courts, derive age_bucket, audit column

In [0]:
#==================================IMPORTS=========================================
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from pyspark.sql.window import *
from typing import *

In [0]:
#============================== SPARK SESSION =====================================
spark = SparkSession.builder \
                .config('spark.sql.shuffle.partitions', 20) \
                .getOrCreate()

In [0]:
#==================================STORAGE PATHs======================================
bronze_cases_path    = "abfss://bronze@saadlsecourtsindia.dfs.core.windows.net/cases/"
bronze_courts_path   = "abfss://bronze@saadlsecourtsindia.dfs.core.windows.net/courts/"
silver_cleaned_path  = "abfss://silver@saadlsecourtsindia.dfs.core.windows.net/cases_cleaned/"
silver_enriched_path = "abfss://silver@saadlsecourtsindia.dfs.core.windows.net/cases_enriched/"

In [0]:
cases_bronze_df  = spark.read.format("delta").load(bronze_cases_path)
courts_bronze_df = spark.read.format("delta").load(bronze_courts_path)

print(f"Bronze cases  count : {cases_bronze_df.count()}")
print(f"Bronze courts count : {courts_bronze_df.count()}")


Bronze cases  count : 3000
Bronze courts count : 50


### Cases : bronze -> silver_cleaned

In [0]:
#================================CASES CLEANED=======================================
# 3. silver_cleaned — Cast, Null Drop, Dedupe, Keep is_deleted

row_rank_window = Window.partitionBy("case_id").orderBy(col("last_modified").desc())

cases_cleaned_df = \
    cases_bronze_df \
    .dropna(subset=["case_id", "case_type", "filing_date", "status", "court_id"]) \
    .withColumn("row_rank", row_number().over(row_rank_window))  \
    .filter(col("row_rank") == 1) \
    .drop("row_rank") \
    .withColumn("silver_transformed_at", current_timestamp())     # audit column


silver_cleaned_count = cases_cleaned_df.count()
print(f"silver_cleaned count : {silver_cleaned_count}")
cases_cleaned_df.printSchema()
cases_cleaned_df.show(truncate=False)

silver_cleaned count : 1000
root
 |-- case_id: integer (nullable = true)
 |-- case_type: string (nullable = true)
 |-- case_subtype: string (nullable = true)
 |-- filing_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- court_id: integer (nullable = true)
 |-- last_modified: timestamp (nullable = true)
 |-- is_deleted: boolean (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- silver_transformed_at: timestamp (nullable = false)

+-------+---------+-----------------+-----------+--------+--------+-------------------+----------+--------------------------+----------------------------------------------------------------------------------------------------+--------------------------+
|case_id|case_type|case_subtype     |filing_date|status  |court_id|last_modified      |is_deleted|bronze_ingested_at        |source_file                                                                                       

In [0]:
#===============================================================================================
# 4. Write silver_cleaned — MERGE on case_id

# check if silver_cleaned delta table already exists
silver_cleaned_exists = DeltaTable.isDeltaTable(spark, silver_cleaned_path)


if silver_cleaned_exists:
    # table exists — merge new/updated records into it
    DeltaTable.forPath(spark, silver_cleaned_path).alias("target") \
        .merge(cases_cleaned_df.alias("source"), "target.case_id = source.case_id") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print("silver_cleaned : MERGE complete")
else:
    # first run — table doesn't exist yet, create it
    cases_cleaned_df.write.format("delta").mode("overwrite").save(silver_cleaned_path)
    print("silver_cleaned : created for the first time")

silver_cleaned : MERGE complete


In [0]:
#=================================================================================
# 5. Read silver_cleaned for enrichment -> remove soft delete -> join courts & cases

#cleaned_for_enrich_df = spark.read.format("delta").load(silver_cleaned_path)  # Load cleaned data from silver_cleaned
#active_cases_df = cleaned_for_enrich_df.filter(col("is_deleted") == False)    # filter out soft-deleted cases

active_cases_df = cases_cleaned_df.filter(col("is_deleted") == False)

enriched_df = active_cases_df.join(courts_bronze_df,on="court_id",how="inner") # Join active_cases_df & courts_bronze_df


#These two column were repeadted and throwing [DELTA_DUPLICATE_COLUMNS_FOUND] Found duplicate column(s) Error
enriched_df = active_cases_df.join(courts_bronze_df, on="court_id", how="inner") \
    .drop(courts_bronze_df["bronze_ingested_at"]) \
    .drop(courts_bronze_df["source_file"])

### `Silver Enriched`

In [0]:
# derive age_bucket from filing_date to today # Business Logic
enriched_df = enriched_df.withColumn("age_years", \
    (datediff(current_date(), col("filing_date")) / 365.25).cast("int")
)

# Case Distribution Logic
enriched_df = enriched_df \
    .withColumn("age_bucket", \
                 when(col("age_years") <  1,  lit("< 1 Year"))  \
                .when(col("age_years") <  3,  lit("1-3 Years")) \
                .when(col("age_years") <  5,  lit("3-5 Years")) \
                .when(col("age_years") < 10,  lit("5-10 Years")) \
                .otherwise( lit("10+ Years") ) \
                ) \
    .drop("age_years") \
    .withColumn("silver_transformed_at", current_timestamp())


print(f"silver_enriched count : {enriched_df.count()}")
enriched_df.show(truncate=False)


silver_enriched count : 1000
+--------+-------+---------+-----------------+-----------+--------+-------------------+----------+--------------------------+----------------------------------------------------------------------------------------------------+--------------------------+--------------------------+-----------+---------------+---------------+-------------+-----------+----------+
|court_id|case_id|case_type|case_subtype     |filing_date|status  |last_modified      |is_deleted|bronze_ingested_at        |source_file                                                                                         |silver_transformed_at     |court_name                |court_level|parent_court_id|bench_name     |state_name   |judge_count|age_bucket|
+--------+-------+---------+-----------------+-----------+--------+-------------------+----------+--------------------------+----------------------------------------------------------------------------------------------------+---------------------

In [0]:
# 7. Write silver_enriched — MERGE on case_id

if DeltaTable.isDeltaTable(spark, silver_enriched_path):
    DeltaTable.forPath(spark, silver_enriched_path).alias("target").merge(
        enriched_df.alias("source"),
        "target.case_id = source.case_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    print("silver_enriched : MERGE complete")
else:
    enriched_df.write.format("delta").mode("overwrite").save(silver_enriched_path)
    print("silver_enriched : first run, table created")


silver_enriched : first run, table created


In [0]:
#======================================================================================
# 8. Validation Checks

print("=== silver_cleaned schema ===")
cases_cleaned_df.printSchema()

print("=== silver_enriched schema ===")
enriched_df.printSchema()

print("=== is_deleted distribution in silver_cleaned ===")
cases_cleaned_df.groupBy("is_deleted").count().show()

print("=== age_bucket distribution in silver_enriched ===")
enriched_df.groupBy("age_bucket").count().orderBy("age_bucket").show()

print("=== court_level distribution in silver_enriched ===")
enriched_df.groupBy("court_level").count().show()

print("=== case_id=1 dedup check (should be Disposed) ===")
cases_cleaned_df.filter(col("case_id") == 1).select("case_id","status","last_modified").show()

=== silver_cleaned schema ===
root
 |-- case_id: integer (nullable = true)
 |-- case_type: string (nullable = true)
 |-- case_subtype: string (nullable = true)
 |-- filing_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- court_id: integer (nullable = true)
 |-- last_modified: timestamp (nullable = true)
 |-- is_deleted: boolean (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- silver_transformed_at: timestamp (nullable = false)

=== silver_enriched schema ===
root
 |-- court_id: integer (nullable = true)
 |-- case_id: integer (nullable = true)
 |-- case_type: string (nullable = true)
 |-- case_subtype: string (nullable = true)
 |-- filing_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- last_modified: timestamp (nullable = true)
 |-- is_deleted: boolean (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- si